[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://drive.google.com/file/d/1jMAJXR9zl7jqX56cdmpXIgZdnt4uqcFW/view?usp=drive_link)

# LLM Evaluation – Custom Metrics

This notebook demonstrates how to define and use custom metrics with Floeval. You can use function-based metrics (no API key required) or LLM-as-judge criteria.

**Objectives**
- Install Floeval and configure credentials (for criteria-based metrics)
- Load evaluation data from a JSON file you provide
- Define a simple `@custom_metric` function (response-only, no API)
- Define a metric that uses `question` and `response`
- Optionally use `criteria()` for LLM-as-judge scoring

## 1. Installation

Install Floeval before running this notebook.

In [ ]:
%pip install floeval>=0.2.0b1

## 2. Configuration Constants

Set the following constants before running. Required only for the optional criteria-based metric (Section 5). Function-based metrics do not require an API key.

**Provider flexibility:** You can use any OpenAI-compatible provider (OpenAI, Azure OpenAI, Anthropic, local models, etc.) — set the appropriate `base_url` and model names for your provider.

**Using FloTorch:** If you want to use FloTorch keys and gateway, obtain credentials from the [FloTorch Console](https://docs.flotorch.cloud/introduction/).

In [ ]:
import getpass
# LLM and API configuration (OpenAI)

OPENAI_BASE_URL = "https://api.openai.com/v1"
OPENAI_API_KEY = getpass.getpass("Enter your API key: ")
OPENAI_CHAT_MODEL = "gpt-4o-mini"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"

## 3. Imports

Import evaluation components, `custom_metric`, and `criteria` for function-based and LLM-judge metrics.

In [ ]:
from pathlib import Path

from floeval import Evaluation, DatasetLoader
from floeval.config.schemas.io.llm import OpenAIProviderConfig
from floeval.api.metrics.custom import custom_metric, criteria

## 4. Define a Simple Custom Metric (Response Only)

The `@custom_metric(threshold=...)` decorator wraps a function into a Floeval metric. This metric uses only `response` and requires no API key.

In [ ]:
@custom_metric(threshold=0.3)
def response_length(response: str) -> float:
    """Score 0–1 based on response length (capped at 100 chars)."""
    return min(len(response) / 100.0, 1.0)

## 5. Define Another Custom Metric (Politeness)

A second custom metric scores politeness by counting polite phrases. Function-based metrics do not require an LLM or API key.

In [ ]:
@custom_metric(threshold=0.5)
def politeness(response: str) -> float:
    """Score 0–1 based on presence of polite phrases."""
    polite_words = ["please", "thank you", "thanks"]
    count = sum(1 for w in polite_words if w in response.lower())
    return min(count / 3.0, 1.0)

## 6. Load Dataset and Run Custom-Only Evaluation (No API Key)

Minimal JSON shape (same as full LLM dataset):

```json
{
  "samples": [ { "user_input": "...", "llm_response": "..." } ]
}
```

**Example file**  
<a href="../datasets/llm_evaluation/sample_llm_custom_metrics.json" download="sample_llm_custom_metrics.json">sample_llm_custom_metrics.json</a>

Provide the dataset path (or upload in Colab), then load and evaluate with custom metrics.


### Resolve Dataset Path

Provide `dataset_path` via file upload in Colab or local JSON path input in Jupyter.


In [ ]:
try:
    from google.colab import files
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    print("Upload your dataset JSON file:")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded.")
    dataset_path = Path(next(iter(uploaded.keys())))
else:
    dataset_path = Path(input("Enter path to dataset JSON file: ").strip().strip('"')).expanduser()


### Load Dataset

Load and validate the dataset with `DatasetLoader.from_file(..., partial_dataset=False)`.


In [ ]:
dataset = DatasetLoader.from_file(dataset_path, partial_dataset=False)
print(f"Dataset loaded from {dataset_path}: {len(dataset.samples)} samples")


### Build the evaluation

Creates an `Evaluation` with the loaded `dataset` and custom metrics only (`response_length`, `politeness`). No `llm_config` is required for this step.


In [ ]:
evaluation = Evaluation(
    dataset=dataset,
    metrics=["response_length", "politeness"],
    default_provider="custom",
)


### Run the evaluation

Calls `evaluation.run()` to compute metrics and fill `results` (aggregate scores and `sample_results`).


In [ ]:
results = evaluation.run()
print("Aggregate scores (no API key used):", results.aggregate_scores)


## 7. Optional: Criteria-Based Metric (LLM-as-Judge)

Use `criteria()` to define an LLM-as-judge rubric. This section requires `llm_config` and an API key.

### Define Criteria Metric

Declare an `empathy` rubric using `criteria()` for LLM-as-judge scoring.


In [ ]:
empathy = criteria(
    name="empathy",
    description="Rate empathy from 0 to 1. Consider acknowledgment, understanding, support.",
    threshold=0.6,
)


### Configure the LLM provider

Builds `OpenAIProviderConfig` (base URL, API key, chat and embedding models) for metrics that call your provider.


In [ ]:
llm_config = OpenAIProviderConfig(
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
    chat_model=OPENAI_CHAT_MODEL,
    embedding_model=OPENAI_EMBEDDING_MODEL,
)


### Build criteria-only evaluation

`Evaluation` using only the `empathy` criteria metric and the same `dataset` as above.


In [ ]:
eval_with_criteria = Evaluation(
    dataset=dataset,
    llm_config=llm_config,
    metrics=[empathy],
)


### Run criteria evaluation

Executes LLM-as-judge scoring and prints aggregate criteria scores.


In [ ]:
criteria_results = eval_with_criteria.run()
print("Criteria (empathy) scores:", criteria_results.aggregate_scores)


## Summary

This notebook demonstrated how to define and use custom metrics with Floeval.

The key components included:

1. **Function-Based Metrics**: The `@custom_metric` decorator was used to define `response_length` and `politeness` metrics that require no API key.
2. **Evaluation Without LLM**: An evaluation was run with custom metrics only, without `llm_config`.
3. **Criteria-Based Metrics**: The `criteria()` helper was used to define an LLM-as-judge metric when an API key is available.
4. **Metric Parameters**: Custom metrics support parameters such as `response`, `question`, `contexts`, `context`, `llm`, and `sample`.

This example showcases both function-based and LLM-as-judge custom metrics in Floeval.